# Comparación de modelos y seguimiento de experimentos en MLflow

En este notebook comparamos varias alternativas de regresión para predecir el límite inferior y superior del salario anual de una vacante. Cada configuración registra sus parámetros, métricas y predicciones en el servidor de MLflow del proyecto. El propósito es elegir, con evidencia reproducible, la familia que pasará al modelado definitivo del notebook 06.

No usamos la prueba final para escoger el modelo: comparamos primero en validación temporal y reservamos el periodo más reciente para evaluarlo una única vez en el notebook 06. La comparación no modifica por sí sola el alias de producción `champion`.

## 1. Alcance, criterios y librerías

Trabajamos exclusivamente con rangos salariales reportados por las vacantes; no mezclamos los estimados por Foorilla en este experimento principal. Seleccionamos por menor MAE promedio de los dos límites y complementamos la decisión con RMSE, R², MAPE, error de amplitud, cobertura e incoherencia entre límites.

La preparación de MLflow reutiliza el experimento configurado, lo restaura si fue eliminado o lo crea si todavía no existe. Al iniciar una nueva ejecución se eliminan únicamente las corridas activas generadas anteriormente por este notebook. Las corridas del modelo definitivo y de otros flujos permanecen disponibles.


In [1]:
from pathlib import Path
from urllib.parse import urldefrag
import json, os, re, sys, warnings
import joblib, mlflow, mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from dotenv import load_dotenv
from IPython.display import display
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
os.environ.setdefault('MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT', 'true')
os.environ.setdefault('MLFLOW_PRINT_MODEL_URLS_ON_CREATION', 'false')
SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA_DIR, ARTIFACT_DIR = (
    ROOT / "data" / "raw" / "foorilla",
    ROOT / "artifacts",
)
EXPERIMENT_DIR = ARTIFACT_DIR / 'comparacion_modelos'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / '.env', override=False)
TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', '').strip().rstrip('/')
if not TRACKING_URI:
    raise EnvironmentError('Configure MLFLOW_TRACKING_URI en el archivo local ml/.env antes de ejecutar el notebook.')
if not TRACKING_URI.startswith(('http://', 'https://')):
    raise ValueError('MLFLOW_TRACKING_URI debe incluir el protocolo http:// o https://.')
EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'salary-prediction')
REGISTERED_MODEL_NAME = os.getenv('MLFLOW_MODEL_NAME', 'salary-predictor')
PRODUCTION_ALIAS = os.getenv('MLFLOW_MODEL_ALIAS', 'champion')
LOG_MODELS_TO_MLFLOW = os.getenv('LOG_MODELS_TO_MLFLOW', 'false').lower() == 'true'
os.environ['MLFLOW_TRACKING_URI'] = TRACKING_URI
os.environ['MLFLOW_EXPERIMENT_NAME'] = EXPERIMENT_NAME
os.environ['MLFLOW_MODEL_NAME'] = REGISTERED_MODEL_NAME
mlflow.set_tracking_uri(TRACKING_URI)
mlflow_client = MlflowClient(tracking_uri=TRACKING_URI)
RESET_NOTEBOOK_RUNS = os.getenv('MLFLOW_RESET_NOTEBOOK_RUNS', 'true').lower() == 'true'
NOTEBOOK_FLOW = 'notebook_05_comparacion_modelos'

def prepare_mlflow_experiment():
    try:
        experiments = mlflow_client.search_experiments(view_type=ViewType.ALL, max_results=10000)
    except Exception as error:
        raise ConnectionError(
            'No fue posible conectar con el servidor definido en MLFLOW_TRACKING_URI. Verifique la configuración y el acceso de red.'
        ) from error

    experiment = next((item for item in experiments if item.name == EXPERIMENT_NAME), None)
    if experiment is None:
        experiment_id = mlflow_client.create_experiment(EXPERIMENT_NAME)
        experiment_action = 'creado'
    else:
        experiment_id = experiment.experiment_id
        if experiment.lifecycle_stage == 'deleted':
            mlflow_client.restore_experiment(experiment_id)
            experiment_action = 'restaurado'
        else:
            experiment_action = 'reutilizado'

    mlflow.set_experiment(experiment_id=experiment_id)
    deleted_runs = 0
    if RESET_NOTEBOOK_RUNS:
        previous_runs = mlflow_client.search_runs(
            experiment_ids=[experiment_id],
            filter_string=f"tags.flujo = '{NOTEBOOK_FLOW}'",
            run_view_type=ViewType.ACTIVE_ONLY,
            max_results=50000,
        )
        for previous_run in previous_runs:
            mlflow_client.delete_run(previous_run.info.run_id)
        deleted_runs = len(previous_runs)
    return experiment_id, experiment_action, deleted_runs

EXPERIMENT_ID, EXPERIMENT_ACTION, DELETED_NOTEBOOK_RUNS = prepare_mlflow_experiment()
print({'python': sys.version.split()[0], 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
       'mlflow': mlflow.__version__, 'lightgbm': __import__('lightgbm').__version__,
       'xgboost': __import__('xgboost').__version__, 'catboost': __import__('catboost').__version__,
       'semilla': SEED, 'conexion_mlflow': 'configurada_desde_entorno', 'experimento': EXPERIMENT_NAME,
       'modelo_registrado': REGISTERED_MODEL_NAME, 'alias_produccion': PRODUCTION_ALIAS,
       'experiment_id': EXPERIMENT_ID, 'estado_experimento': EXPERIMENT_ACTION, 'corridas_anteriores_eliminadas': DELETED_NOTEBOOK_RUNS, 'registrar_modelos_intermedios': LOG_MODELS_TO_MLFLOW})

{'python': '3.14.4', 'pandas': '3.0.5', 'sklearn': '1.8.0', 'mlflow': '3.16.0', 'lightgbm': '4.7.0', 'xgboost': '3.4.1', 'catboost': '1.2.10', 'semilla': 42, 'conexion_mlflow': 'configurada_desde_entorno', 'experimento': 'salary-prediction', 'modelo_registrado': 'salary-predictor', 'alias_produccion': 'champion', 'experiment_id': '4', 'estado_experimento': 'reutilizado', 'corridas_anteriores_eliminadas': 38, 'registrar_modelos_intermedios': False}


## 2. Integración y limpieza de la muestra

Conservo la lógica del EDA: priorizo el corte más reciente por identificador, retiro republicaciones por URL, empresa, cargo y ubicación, y conservo rangos positivos y ordenados. También retiro extremos mediante la cerca exterior de Tukey sobre el punto medio logarítmico.

In [2]:
files = sorted(DATA_DIR.glob('jobs_*.csv'))
assert files, 'No se encontraron archivos jobs_*.csv en data/'
parts = []
for priority, path in enumerate(files):
    frame = pd.read_csv(path, low_memory=False)
    frame['_priority'] = priority
    parts.append(frame)
raw = pd.concat(parts, ignore_index=True)
raw['published'] = pd.to_datetime(raw['published'], errors='coerce', utc=True)
raw['_complete'] = raw.notna().sum(axis=1)
by_id = raw.sort_values(['id', '_priority', '_complete', 'published']).drop_duplicates('id', keep='last').copy()

def norm(values):
    return (values.fillna('').astype(str).str.lower().str.normalize('NFKD')
            .str.encode('ascii', errors='ignore').str.decode('ascii')
            .str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip())

url = by_id.apply_url.fillna('').astype(str).map(lambda value: urldefrag(value)[0].rstrip('/').lower())
by_id['_signature'] = url + '|' + norm(by_id.company) + '|' + norm(by_id.title) + '|' + norm(by_id.location)
by_id['_salary_info'] = by_id[['salary_min', 'salary_max', 'salary_min_usd', 'salary_max_usd']].notna().sum(axis=1)
dedup = by_id.sort_values(['_signature', '_salary_info', '_complete', 'published']).drop_duplicates('_signature', keep='last').copy()
for side in ['min', 'max']:
    dedup[f'y_{side}_usd'] = pd.to_numeric(dedup[f'salary_{side}_usd'], errors='coerce')
    dedup[f'{side}_reported'] = pd.to_numeric(dedup[f'salary_{side}'], errors='coerce').notna()
dedup['target_source'] = np.select([dedup.min_reported & dedup.max_reported,
                                    dedup.min_reported | dedup.max_reported],
                                   ['reportado', 'híbrido'], default='estimado')
valid = (dedup.y_min_usd.notna() & dedup.y_max_usd.notna() & (dedup.y_min_usd > 0) &
         (dedup.y_max_usd > 0) & (dedup.y_min_usd <= dedup.y_max_usd))
model_df = dedup.loc[valid].copy()
log_mid = np.log1p((model_df.y_min_usd + model_df.y_max_usd) / 2)
q1, q3 = log_mid.quantile([.25, .75])
model_df = model_df.loc[log_mid.between(q1 - 3 * (q3-q1), q3 + 3 * (q3-q1))].copy()
display(pd.Series({'filas_integradas': len(raw), 'identificadores_unicos': len(by_id),
                   'vacantes_deduplicadas': len(dedup), 'muestra_modelado': len(model_df),
                   'rangos_reportados': int(model_df.target_source.eq('reportado').sum())}))

filas_integradas          338667
identificadores_unicos    338281
vacantes_deduplicadas     287415
muestra_modelado          263823
rangos_reportados          55002
dtype: int64

## 3. Tipos de datos, variables y validación temporal

Tomamos como base la revisión de campos realizada en `02_eda.ipynb`. Las variables categóricas de texto requieren codificación, las numéricas requieren imputación y las etiquetas múltiples se convierten en indicadores binarios. Solo empleamos características disponibles al publicar: cargo, país, región, experiencia, modalidad, empresa, agencia, años, fecha y habilidades en `tags`. Excluimos salarios, conversiones, URL, identificadores y `expired` para evitar fuga de información. Usamos 70% para entrenamiento, 15% para validación y 15% para prueba final, respetando el orden temporal.

In [3]:
SKILLS = {'python':'python', 'sql':'sql', 'aws':'aws', 'azure':'azure', 'gcp':'gcp', 'spark':'spark',
          'docker':'docker', 'kubernetes':'kubernetes', 'machine_learning':'machine learning',
          'pytorch':'pytorch', 'tensorflow':'tensorflow', 'tableau':'tableau', 'power_bi':'power bi'}
def prepare_features(df):
    out = pd.DataFrame(index=df.index)
    out['title'] = norm(df.title).replace('', 'desconocido')
    out['country'] = df.countries.fillna('desconocido').astype(str).str.split('|').str[0].str.strip().replace('', 'desconocido')
    out['region'] = df.regions.fillna('desconocido').astype(str).str.split('|').str[0].str.strip().replace('', 'desconocido')
    out['experience_level'] = df.experience_level.fillna('desconocido').astype(str)
    remote = df.has_remote.fillna(False).astype(bool)
    work = pd.to_numeric(df.work_mode, errors='coerce')
    out['work_mode'] = np.select([~remote, work.eq(1), work.eq(2), work.eq(3)],
                                 ['presencial', 'híbrido', 'remoto', 'remoto_global'], default='remoto_sin_detalle')
    out['company'] = norm(df.company).replace('', 'desconocido')
    out['company_is_agency'] = df.company_is_agency.fillna(False).astype(int)
    out['experience_years'] = pd.to_numeric(df.experience_years, errors='coerce').where(lambda x: x.between(0, 50))
    out['experience_years_missing'] = out.experience_years.isna().astype(int)
    tags = df.tags.fillna('').astype(str).str.lower()
    for name, token in SKILLS.items(): out[f'skill_{name}'] = tags.str.contains(re.escape(token), regex=True).astype(int)
    out['published_year'], out['published_month'] = df.published.dt.year, df.published.dt.month
    return out

X = prepare_features(model_df)
TARGETS = ['y_min_usd', 'y_max_usd']
assert not [column for column in X if 'salary' in column.lower()], 'Hay fuga salarial en las variables'
ordered = model_df.loc[model_df.target_source.eq('reportado')].sort_values('published', na_position='first').index
n = len(ordered); cut_train, cut_val = int(.70*n), int(.85*n)
train_idx, val_idx, test_idx = ordered[:cut_train], ordered[cut_train:cut_val], ordered[cut_val:]
display(pd.DataFrame({'particion':['entrenamiento', 'validacion', 'prueba_final'],
                      'n':[len(train_idx), len(val_idx), len(test_idx)],
                      'inicio':[model_df.loc[x].published.min() for x in [train_idx, val_idx, test_idx]],
                      'fin':[model_df.loc[x].published.max() for x in [train_idx, val_idx, test_idx]]}))
feature_contract = pd.DataFrame([
    {'campo_origen':'title','tipo_origen':'texto','variables_modelo':'title','transformacion':'normalización + TargetEncoder'},
    {'campo_origen':'countries / regions','tipo_origen':'texto multivalor','variables_modelo':'country, region','transformacion':'primera ubicación + TargetEncoder'},
    {'campo_origen':'experience_level','tipo_origen':'categórica','variables_modelo':'experience_level','transformacion':'imputación + TargetEncoder'},
    {'campo_origen':'experience_years','tipo_origen':'numérica','variables_modelo':'experience_years, experience_years_missing','transformacion':'rango 0-50 + mediana + bandera'},
    {'campo_origen':'has_remote / work_mode','tipo_origen':'booleano + código','variables_modelo':'work_mode','transformacion':'modalidad homologada + TargetEncoder'},
    {'campo_origen':'company / company_is_agency','tipo_origen':'texto + booleano','variables_modelo':'company, company_is_agency','transformacion':'TargetEncoder + binaria'},
    {'campo_origen':'tags','tipo_origen':'texto multietiqueta','variables_modelo':'skill_*','transformacion':'vocabulario controlado binario'},
    {'campo_origen':'published','tipo_origen':'fecha','variables_modelo':'published_year, published_month','transformacion':'componentes numéricos'}])
display(feature_contract)

,particion,n,inicio,fin
0,entrenamiento,38501,2025-01-01 00:13:20+00:00,2026-03-04 00:00:00+00:00
1,validacion,8250,2026-03-04 00:00:00+00:00,2026-06-01 23:11:07+00:00
2,prueba_final,8251,2026-06-02 00:00:00+00:00,2026-09-09 13:30:04+00:00


,campo_origen,tipo_origen,variables_modelo,transformacion
0,title,texto,title,normalización + TargetEncoder
1,countries / regions,texto multivalor,"country, region",primera ubicación + TargetEncoder
2,experience_level,categórica,experience_level,imputación + TargetEncoder
3,experience_years,numérica,"experience_years, experience_years_missing",rango 0-50 + mediana + bandera
4,has_remote / work_mode,booleano + código,work_mode,modalidad homologada + TargetEncoder
5,company / company_is_agency,texto + booleano,"company, company_is_agency",TargetEncoder + binaria
6,tags,texto multietiqueta,skill_*,vocabulario controlado binario
7,published,fecha,"published_year, published_month",componentes numéricos


## 4. Preparación común y función de evaluación

Para Ridge y los modelos de árboles convertimos las variables categóricas mediante `TargetEncoder` con ajuste cruzado e imputamos las variables numéricas con la mediana. Las categorías nuevas reciben la media global aprendida en entrenamiento. CatBoost conserva las categorías como texto porque incorpora un tratamiento nativo para este tipo de variables. En todos los casos entrenamos un modelo para Y1 y otro para Y2 y ordenamos el resultado para evitar rangos invertidos.

Comparamos una línea base, un modelo lineal, cuatro familias incluidas en scikit-learn y tres implementaciones especializadas de boosting: LightGBM, XGBoost y CatBoost. Para que el experimento sea amplio pero reproducible, declaramos primero el rango de cada hiperparámetro y después tomamos una muestra determinística de configuraciones. El número de combinaciones por modelo puede ampliarse con `N_ITER_SEARCH` sin cambiar la lógica del notebook.

In [4]:
cat_cols = ['title', 'country', 'region', 'experience_level', 'work_mode', 'company']
num_cols = [column for column in X.columns if column not in cat_cols]
preprocessor = ColumnTransformer([
    ('categoricas', Pipeline([('imputar', SimpleImputer(strategy='most_frequent')),
                              ('codificar', TargetEncoder(target_type='continuous', smooth='auto', cv=5,
                                                          shuffle=True, random_state=SEED))]), cat_cols),
    ('numericas', SimpleImputer(strategy='median'), num_cols)], verbose_feature_names_out=False)

N_ITER_SEARCH = int(os.getenv('N_ITER_SEARCH', '5'))
MODEL_SPECS = {
    'baseline_mediana': {
        'family':'baseline', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'strategy':'median'}, 'search':{}},
    'ridge': {
        'family':'lineal', 'preprocessing':'TargetEncoder + estandarización', 'scale':True,
        'fixed':{}, 'search':{'alpha':[0.1, 1.0, 10.0, 100.0]}},
    'hist_gradient_boosting': {
        'family':'boosting sklearn', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'loss':'absolute_error'},
        'search':{'learning_rate':[0.03, 0.05, 0.08], 'max_iter':[180, 260, 360],
                  'max_leaf_nodes':[15, 31, 63], 'min_samples_leaf':[10, 20, 40],
                  'l2_regularization':[0.5, 1.0, 3.0]}},
    'gradient_boosting': {
        'family':'boosting sklearn', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'loss':'huber'},
        'search':{'learning_rate':[0.03, 0.05, 0.08], 'n_estimators':[140, 220, 320],
                  'max_depth':[2, 3, 4], 'min_samples_leaf':[5, 10, 20],
                  'subsample':[0.8, 1.0]}},
    'random_forest': {
        'family':'bagging', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{},
        'search':{'n_estimators':[100, 200, 300], 'max_depth':[3, 6, 9],
                  'min_samples_leaf':[2, 4, 8], 'max_features':[0.5, 0.6, 0.7],
                  'max_samples':[0.75, 0.9]}},
    'extra_trees': {
        'family':'bagging', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'bootstrap':True},
        'search':{'n_estimators':[100, 200, 300], 'max_depth':[3, 6, 9],
                  'min_samples_leaf':[2, 4, 8], 'max_features':[0.5, 0.6, 0.7],
                  'max_samples':[0.75, 0.9]}},
    'lightgbm': {
        'family':'boosting especializado', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'objective':'regression_l1', 'verbosity':-1, 'subsample_freq':1},
        'search':{'n_estimators':[400, 600, 800], 'learning_rate':[0.03, 0.05, 0.07],
                  'num_leaves':[31, 63, 95], 'min_child_samples':[15, 30, 60],
                  'subsample':[0.8, 0.9, 1.0], 'colsample_bytree':[0.8, 0.9, 1.0],
                  'reg_lambda':[1.0, 3.0, 6.0], 'reg_alpha':[0.0, 0.25, 0.75]}},
    'xgboost': {
        'family':'boosting especializado', 'preprocessing':'TargetEncoder', 'scale':False,
        'fixed':{'objective':'reg:squarederror', 'tree_method':'hist'},
        'search':{'n_estimators':[300, 500, 700], 'learning_rate':[0.025, 0.04, 0.06],
                  'max_depth':[5, 7, 9], 'min_child_weight':[3, 7, 12],
                  'subsample':[0.8, 1.0], 'colsample_bytree':[0.8, 1.0],
                  'reg_lambda':[1.0, 3.0, 8.0]}},
    'catboost': {
        'family':'boosting especializado', 'preprocessing':'categóricas nativas', 'scale':False,
        'fixed':{'loss_function':'MAE', 'verbose':False, 'allow_writing_files':False},
        'search':{'iterations':[350, 550, 750], 'learning_rate':[0.025, 0.04, 0.06],
                  'depth':[6, 8, 10], 'l2_leaf_reg':[3.0, 6.0, 10.0],
                  'random_strength':[0.5, 1.0, 2.0]}}
}

def readable(values):
    return json.dumps(values, ensure_ascii=False, default=str)

catalog_rows=[];search_summary=[]
for name,spec in MODEL_SPECS.items():
    for parameter,value in spec['fixed'].items():
        catalog_rows.append({'modelo':name,'familia':spec['family'],'preprocesamiento':spec['preprocessing'],
                             'tipo':'fijo','hiperparámetro':parameter,'valores_considerados':readable(value)})
    for parameter,values in spec['search'].items():
        catalog_rows.append({'modelo':name,'familia':spec['family'],'preprocesamiento':spec['preprocessing'],
                             'tipo':'búsqueda','hiperparámetro':parameter,'valores_considerados':readable(values)})
    combinations=int(np.prod([len(values) for values in spec['search'].values()])) if spec['search'] else 1
    search_summary.append({'modelo':name,'parámetros_fijos':len(spec['fixed']),
                           'hiperparámetros_ajustados':len(spec['search']),
                           'combinaciones_posibles':combinations,
                           'configuraciones_evaluadas':min(N_ITER_SEARCH,combinations)})
hyperparameter_catalog=pd.DataFrame(catalog_rows)
search_summary=pd.DataFrame(search_summary)
display(hyperparameter_catalog.style.set_properties(subset=['valores_considerados'],**{'text-align':'left'}))
display(search_summary)
print(f'Conclusión del espacio de búsqueda: evaluaremos hasta {N_ITER_SEARCH} configuraciones por modelo '
      f'mediante muestreo aleatorio reproducible con semilla {SEED}.')
numeric_check=clone(preprocessor).fit_transform(X.loc[train_idx].head(5000),
                                                np.log1p(model_df.loc[train_idx].head(5000).y_min_usd))
display(pd.Series({'filas_verificadas':numeric_check.shape[0],'columnas_numericas':numeric_check.shape[1],
                   'tipo_resultante':str(numeric_check.dtype),'valores_no_finitos':int((~np.isfinite(numeric_check)).sum())}))

def metricas(y_true, raw_prediction):
    prediction = np.sort(np.asarray(raw_prediction), axis=1)
    result = {}
    for position, label in enumerate(['min', 'max']):
        result[f'mae_{label}'] = mean_absolute_error(y_true[:, position], prediction[:, position])
        result[f'rmse_{label}'] = mean_squared_error(y_true[:, position], prediction[:, position]) ** .5
        result[f'mape_{label}'] = mean_absolute_percentage_error(y_true[:, position], prediction[:, position])
        result[f'r2_{label}'] = r2_score(y_true[:, position], prediction[:, position])
    result['mae_promedio'] = (result['mae_min'] + result['mae_max']) / 2
    result['mae_amplitud'] = mean_absolute_error(y_true[:,1]-y_true[:,0], prediction[:,1]-prediction[:,0])
    result['cobertura_intervalo'] = float(np.mean((y_true[:,0] >= prediction[:,0]) & (y_true[:,1] <= prediction[:,1])))
    result['incoherencia_raw'] = float(np.mean(raw_prediction[:,0] > raw_prediction[:,1]))
    return result, prediction

def build_estimator(name, params):
    if name == 'baseline_mediana': return DummyRegressor(**params)
    if name == 'ridge': return Ridge(**params)
    if name == 'hist_gradient_boosting': return HistGradientBoostingRegressor(**params, random_state=SEED)
    if name == 'gradient_boosting': return GradientBoostingRegressor(**params, random_state=SEED)
    if name == 'random_forest': return RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)
    if name == 'extra_trees': return ExtraTreesRegressor(**params, n_jobs=-1, random_state=SEED)
    if name == 'lightgbm': return LGBMRegressor(**params, n_jobs=-1, random_state=SEED)
    if name == 'xgboost': return XGBRegressor(**params, n_jobs=-1, random_state=SEED)
    if name == 'catboost': return CatBoostRegressor(**params, cat_features=cat_cols,
                                                     thread_count=-1, random_seed=SEED)
    raise KeyError(name)

def parameter_candidates(name):
    spec = MODEL_SPECS[name]
    if not spec['search']:
        return [dict(spec['fixed'])]
    total = int(np.prod([len(values) for values in spec['search'].values()]))
    sampled = list(ParameterSampler(spec['search'], n_iter=min(N_ITER_SEARCH, total), random_state=SEED))
    return [{**spec['fixed'], **candidate} for candidate in sampled]

planned_configurations=[]
for model_name in MODEL_SPECS:
    for number,params in enumerate(parameter_candidates(model_name),start=1):
        planned_configurations.append({'modelo':model_name,
            'configuracion':f'{model_name}__{number:02d}','hiperparametros':readable(params)})
planned_configurations=pd.DataFrame(planned_configurations)
display(planned_configurations.style.set_properties(subset=['hiperparametros'],**{'text-align':'left'}))
print('Las configuraciones anteriores son exactamente las que se evaluarán en las siguientes pruebas.')

def ejecutar_configuracion(name, params, configuration, train_rows, eval_rows, stage='validacion'):
    spec = MODEL_SPECS[name]
    fitted, predictions = [], []
    with mlflow.start_run(run_name=f'{stage}-{configuration}') as run:
        mlflow.set_tags({'flujo':'notebook_05_comparacion_modelos',
                         'estado_seleccion':'en_evaluacion',
                         'modelo_registrado_objetivo':REGISTERED_MODEL_NAME,
                         'alias_produccion_objetivo':PRODUCTION_ALIAS,
                         'etapa':stage})
        mlflow.log_params({'modelo': name, 'familia':spec['family'], 'semilla': SEED, 'n_train': len(train_rows),
                           'n_eval': len(eval_rows), 'particion': 'temporal_70_15_15',
                           'objetivo': 'rangos_reportados', 'transformacion_y': 'log1p',
                           'preprocesamiento':spec['preprocessing'],
                           **{f'hp_{k}':v for k,v in params.items()}})
        for target in TARGETS:
            estimator = build_estimator(name, params)
            if name == 'catboost':
                fitted_model = estimator.fit(X.loc[train_rows], np.log1p(model_df.loc[train_rows, target]))
            else:
                steps=[('preprocesamiento',clone(preprocessor))]
                if spec['scale']: steps.append(('estandarizacion',StandardScaler()))
                steps.append(('modelo',estimator))
                fitted_model=Pipeline(steps).fit(X.loc[train_rows], np.log1p(model_df.loc[train_rows, target]))
            fitted.append(fitted_model)
            predictions.append(np.expm1(fitted_model.predict(X.loc[eval_rows])))
        raw_prediction = np.column_stack(predictions)
        y_true = model_df.loc[eval_rows, TARGETS].to_numpy()
        scores, ordered_prediction = metricas(y_true, raw_prediction)
        mlflow.log_metrics(scores)
        output = EXPERIMENT_DIR / f'predicciones_{stage}_{configuration}.csv'
        pd.DataFrame({'id':model_df.loc[eval_rows,'id'].astype(str), 'y_min':y_true[:,0], 'y_max':y_true[:,1],
                      'pred_min':ordered_prediction[:,0], 'pred_max':ordered_prediction[:,1]}).to_csv(output, index=False)
        mlflow.log_artifact(output)
        if LOG_MODELS_TO_MLFLOW:
            for target, fitted_model in zip(TARGETS, fitted):
                mlflow.sklearn.log_model(fitted_model, name=f'{target}_{configuration}',
                    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE)
        return {'modelo':name, 'familia':spec['family'], 'configuracion':configuration,
                'parametros':readable(params), 'etapa':stage, 'run_id':run.info.run_id, **scores}, fitted

def probar_modelo(name):
    partial=[]
    for number, params in enumerate(parameter_candidates(name), start=1):
        configuration=f'{name}__{number:02d}'
        result,_=ejecutar_configuracion(name,params,configuration,train_idx,val_idx)
        partial.append(result)
    best=pd.DataFrame(partial).sort_values('mae_promedio').iloc[0]
    print(f"Conclusión - {name}: se probaron {len(partial)} configuraciones. "
          f"La mejor fue {best.configuracion}, con MAE promedio USD {best.mae_promedio:,.0f} "
          f"y R² mínimo/máximo {best.r2_min:.3f}/{best.r2_max:.3f}.")
    return partial

,modelo,familia,preprocesamiento,tipo,hiperparámetro,valores_considerados
0,baseline_mediana,baseline,TargetEncoder,fijo,strategy,"""median"""
1,ridge,lineal,TargetEncoder + estandarización,búsqueda,alpha,"[0.1, 1.0, 10.0, 100.0]"
2,hist_gradient_boosting,boosting sklearn,TargetEncoder,fijo,loss,"""absolute_error"""
3,hist_gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,learning_rate,"[0.03, 0.05, 0.08]"
4,hist_gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,max_iter,"[180, 260, 360]"
5,hist_gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,max_leaf_nodes,"[15, 31, 63]"
6,hist_gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,min_samples_leaf,"[10, 20, 40]"
7,hist_gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,l2_regularization,"[0.5, 1.0, 3.0]"
8,gradient_boosting,boosting sklearn,TargetEncoder,fijo,loss,"""huber"""
9,gradient_boosting,boosting sklearn,TargetEncoder,búsqueda,learning_rate,"[0.03, 0.05, 0.08]"


,modelo,parámetros_fijos,hiperparámetros_ajustados,combinaciones_posibles,configuraciones_evaluadas
0,baseline_mediana,1,0,1,1
1,ridge,0,1,4,4
2,hist_gradient_boosting,1,5,243,5
3,gradient_boosting,1,5,162,5
4,random_forest,0,5,162,5
5,extra_trees,1,5,162,5
6,lightgbm,3,8,6561,5
7,xgboost,2,7,972,5
8,catboost,3,5,243,5


Conclusión del espacio de búsqueda: evaluaremos hasta 5 configuraciones por modelo mediante muestreo aleatorio reproducible con semilla 42.


filas_verificadas        5000
columnas_numericas         24
tipo_resultante       float64
valores_no_finitos          0
dtype: object

,modelo,configuracion,hiperparametros
0,baseline_mediana,baseline_mediana__01,"{""strategy"": ""median""}"
1,ridge,ridge__01,"{""alpha"": 0.1}"
2,ridge,ridge__02,"{""alpha"": 1.0}"
3,ridge,ridge__03,"{""alpha"": 10.0}"
4,ridge,ridge__04,"{""alpha"": 100.0}"
5,hist_gradient_boosting,hist_gradient_boosting__01,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 10, ""max_leaf_nodes"": 63, ""max_iter"": 360, ""learning_rate"": 0.03, ""l2_regularization"": 0.5}"
6,hist_gradient_boosting,hist_gradient_boosting__02,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 10, ""max_leaf_nodes"": 63, ""max_iter"": 180, ""learning_rate"": 0.03, ""l2_regularization"": 0.5}"
7,hist_gradient_boosting,hist_gradient_boosting__03,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 40, ""max_leaf_nodes"": 63, ""max_iter"": 260, ""learning_rate"": 0.08, ""l2_regularization"": 1.0}"
8,hist_gradient_boosting,hist_gradient_boosting__04,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 20, ""max_leaf_nodes"": 63, ""max_iter"": 260, ""learning_rate"": 0.08, ""l2_regularization"": 3.0}"
9,hist_gradient_boosting,hist_gradient_boosting__05,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 20, ""max_leaf_nodes"": 31, ""max_iter"": 360, ""learning_rate"": 0.08, ""l2_regularization"": 3.0}"


Las configuraciones anteriores son exactamente las que se evaluarán en las siguientes pruebas.


## 5. Línea base: mediana histórica

La línea base predice la mediana del entrenamiento para todas las vacantes. Es el mínimo que debe superar un modelo útil; de otro modo no justificaría su complejidad.

In [5]:
resultados = probar_modelo('baseline_mediana')

Conclusión - baseline_mediana: se probaron 1 configuraciones. La mejor fue baseline_mediana__01, con MAE promedio USD 55,929 y R² mínimo/máximo -0.013/-0.000.


## 6. Prueba 1: Ridge

Ridge establece una referencia lineal regularizada. Después de convertir todas las variables a números, estandarizamos las columnas para que la penalización sea comparable. Probamos varios niveles de penalización para identificar si las relaciones principales pueden explicarse sin ensambles de árboles.

In [6]:
resultados.extend(probar_modelo('ridge'))

Conclusión - ridge: se probaron 4 configuraciones. La mejor fue ridge__01, con MAE promedio USD 32,360 y R² mínimo/máximo 0.537/0.581.


## 7. Prueba 2: HistGradientBoosting

HistGradientBoosting usa boosting de árboles por histogramas. Variamos la tasa de aprendizaje, el número de iteraciones, el tamaño del árbol y la regularización para controlar el equilibrio entre capacidad y generalización.

In [7]:
resultados.extend(probar_modelo('hist_gradient_boosting'))

Conclusión - hist_gradient_boosting: se probaron 5 configuraciones. La mejor fue hist_gradient_boosting__04, con MAE promedio USD 27,969 y R² mínimo/máximo 0.602/0.662.


## 8. Prueba 3: Gradient Boosting

Gradient Boosting clásico construye árboles pequeños de manera secuencial para corregir los errores del ensamble anterior. Empleamos pérdida Huber para reducir la influencia de observaciones alejadas sin eliminar la señal salarial.

In [8]:
resultados.extend(probar_modelo('gradient_boosting'))

Conclusión - gradient_boosting: se probaron 5 configuraciones. La mejor fue gradient_boosting__01, con MAE promedio USD 27,996 y R² mínimo/máximo 0.607/0.671.


## 9. Prueba 4: Random Forest

Random Forest entrena muchos árboles sobre muestras y variables aleatorias. Lo probamos como alternativa robusta para verificar si el boosting aporta una mejora. El espacio de búsqueda limita la profundidad, exige un mínimo de observaciones por hoja y utiliza submuestras de filas y variables. Estas restricciones controlan el sobreajuste y mantienen un presupuesto comparable con los demás modelos.


In [9]:
resultados.extend(probar_modelo('random_forest'))

Conclusión - random_forest: se probaron 5 configuraciones. La mejor fue random_forest__03, con MAE promedio USD 29,310 y R² mínimo/máximo 0.583/0.653.


## 10. Prueba 5: Extra Trees

Extra Trees añade más aleatoriedad al elegir los puntos de corte. Lo utilizamos para contrastar otro ensamble de árboles frente al boosting. La profundidad, el tamaño mínimo de hoja y el muestreo se controlan con los mismos criterios de regularización usados en Random Forest, sin retirar resultados ni modificar la métrica de comparación.


In [10]:
resultados.extend(probar_modelo('extra_trees'))

Conclusión - extra_trees: se probaron 5 configuraciones. La mejor fue extra_trees__01, con MAE promedio USD 32,571 y R² mínimo/máximo 0.527/0.576.


## 11. Prueba 6: LightGBM

LightGBM construye árboles por hojas y está optimizado para datos tabulares. Probamos número de hojas, tasa de aprendizaje, cantidad de árboles, muestreo de filas y columnas y regularización L1/L2. Conservamos el mismo `TargetEncoder` y el mismo número máximo de configuraciones que en las demás familias, de manera que la comparación utilice las mismas variables, particiones y presupuesto de búsqueda.


In [11]:
resultados.extend(probar_modelo('lightgbm'))

Conclusión - lightgbm: se probaron 5 configuraciones. La mejor fue lightgbm__03, con MAE promedio USD 27,556 y R² mínimo/máximo 0.609/0.669.


## 12. Prueba 7: XGBoost

XGBoost es otro ensamble secuencial de árboles. Su espacio de búsqueda controla profundidad, peso mínimo de los nodos, muestreo y penalización L2. Esto permite contrastarlo con LightGBM sin asumir que una implementación será mejor para estos datos.

In [12]:
resultados.extend(probar_modelo('xgboost'))

Conclusión - xgboost: se probaron 5 configuraciones. La mejor fue xgboost__05, con MAE promedio USD 27,699 y R² mínimo/máximo 0.611/0.672.


## 13. Prueba 8: CatBoost

CatBoost recibe directamente las variables categóricas y calcula sus representaciones durante el entrenamiento. Por esta razón no aplicamos `TargetEncoder` en este candidato. Variamos profundidad, iteraciones, tasa de aprendizaje y regularización, manteniendo la misma partición temporal y las mismas métricas de los demás modelos.

In [13]:
resultados.extend(probar_modelo('catboost'))

Conclusión - catboost: se probaron 5 configuraciones. La mejor fue catboost__03, con MAE promedio USD 28,441 y R² mínimo/máximo 0.596/0.649.


## 14. Mejor configuración por modelo y selección global

Primero escogemos la configuración con menor MAE promedio dentro de cada modelo y después comparamos únicamente esos ganadores. La métrica principal continúa siendo el MAE promedio de validación temporal. Consideramos equivalentes en términos prácticos las alternativas cuya diferencia no supera el 1% respecto al mejor promedio; dentro de ese grupo priorizamos el menor `mae_max`, porque el límite superior ha mostrado el mayor error y es especialmente importante para el rango salarial. Si LightGBM pertenece al grupo equivalente y además obtiene el menor `mae_max`, se conserva como modelo definitivo por su compatibilidad con el pipeline productivo. La tabla mantiene visibles todos los resultados y los hiperparámetros exactos.


In [14]:
all_results=pd.DataFrame(resultados).sort_values(['modelo','mae_promedio']).reset_index(drop=True)
best_by_model=(all_results.sort_values(['mae_promedio','mae_max']).groupby('modelo',as_index=False).first()
               .sort_values(['mae_promedio','mae_max']).reset_index(drop=True))
columns=['modelo','configuracion','parametros','mae_min','mae_max','mae_promedio','rmse_min','rmse_max',
         'r2_min','r2_max','mae_amplitud','cobertura_intervalo','incoherencia_raw','run_id']
display(best_by_model[columns].style.format({'mae_min':'USD {:,.0f}','mae_max':'USD {:,.0f}',
    'mae_promedio':'USD {:,.0f}','rmse_min':'USD {:,.0f}','rmse_max':'USD {:,.0f}',
    'mae_amplitud':'USD {:,.0f}','r2_min':'{:.3f}','r2_max':'{:.3f}',
    'cobertura_intervalo':'{:.1%}','incoherencia_raw':'{:.2%}'}))

PRACTICAL_TOLERANCE=0.01
best_mean=float(best_by_model.mae_promedio.min())
eligible=best_by_model.loc[best_by_model.mae_promedio<=best_mean*(1+PRACTICAL_TOLERANCE)].copy()
eligible['diferencia_relativa_mae']=(eligible.mae_promedio-best_mean)/best_mean
eligible=eligible.sort_values(['mae_max','mae_promedio']).reset_index(drop=True)
lightgbm_eligible=eligible.loc[eligible.modelo.eq('lightgbm')]
if not lightgbm_eligible.empty and lightgbm_eligible.iloc[0].mae_max<=eligible.mae_max.min()+1e-9:
    winner_row=lightgbm_eligible.iloc[0]
    selection_reason=('equivalencia práctica del 1%, menor MAE del límite superior y '
                      'compatibilidad con el pipeline productivo')
else:
    winner_row=eligible.iloc[0]
    selection_reason='equivalencia práctica del 1% y menor MAE del límite superior'
winner=str(winner_row.modelo)
winner_params=json.loads(winner_row.parametros)
display(eligible[['modelo','configuracion','mae_promedio','mae_max','diferencia_relativa_mae']].style.format({
    'mae_promedio':'USD {:,.0f}','mae_max':'USD {:,.0f}','diferencia_relativa_mae':'{:.2%}'}))
print(f'Modelo seleccionado para el notebook 06: {winner} ({winner_row.configuracion}).')
print(f'Criterio aplicado: {selection_reason}.')
print(f'MAE promedio de validación: USD {winner_row.mae_promedio:,.0f}; '
      f'MAE del límite superior: USD {winner_row.mae_max:,.0f}; '
      f'R² mínimo/máximo: {winner_row.r2_min:.3f}/{winner_row.r2_max:.3f}.')


,modelo,configuracion,parametros,mae_min,mae_max,mae_promedio,rmse_min,rmse_max,r2_min,r2_max,mae_amplitud,cobertura_intervalo,incoherencia_raw,run_id
0,lightgbm,lightgbm__03,"{""objective"": ""regression_l1"", ""verbosity"": -1, ""subsample_freq"": 1, ""subsample"": 0.8, ""reg_lambda"": 6.0, ""reg_alpha"": 0.25, ""num_leaves"": 63, ""n_estimators"": 600, ""min_child_samples"": 15, ""learning_rate"": 0.05, ""colsample_bytree"": 1.0}","USD 23,104","USD 32,009","USD 27,556","USD 37,023","USD 52,055",0.609,0.669,"USD 21,661",12.5%,1.35%,4018203a81b7420da58caacea28c2d9b
1,xgboost,xgboost__05,"{""objective"": ""reg:squarederror"", ""tree_method"": ""hist"", ""subsample"": 0.8, ""reg_lambda"": 8.0, ""n_estimators"": 700, ""min_child_weight"": 12, ""max_depth"": 7, ""learning_rate"": 0.025, ""colsample_bytree"": 0.8}","USD 23,219","USD 32,179","USD 27,699","USD 36,934","USD 51,847",0.611,0.672,"USD 21,016",12.2%,1.44%,1f383ae1c65d43e4b700cff140102ce6
2,hist_gradient_boosting,hist_gradient_boosting__04,"{""loss"": ""absolute_error"", ""min_samples_leaf"": 20, ""max_leaf_nodes"": 63, ""max_iter"": 260, ""learning_rate"": 0.08, ""l2_regularization"": 3.0}","USD 23,393","USD 32,546","USD 27,969","USD 37,353","USD 52,591",0.602,0.662,"USD 21,974",13.1%,1.39%,7419ea5fec5842928e6350e9d095734b
3,gradient_boosting,gradient_boosting__01,"{""loss"": ""huber"", ""subsample"": 0.8, ""n_estimators"": 220, ""min_samples_leaf"": 20, ""max_depth"": 4, ""learning_rate"": 0.08}","USD 23,575","USD 32,418","USD 27,996","USD 37,141","USD 51,921",0.607,0.671,"USD 21,659",12.7%,1.14%,53227a0cd4394bcf899d0da31790b83d
4,catboost,catboost__03,"{""loss_function"": ""MAE"", ""verbose"": false, ""allow_writing_files"": false, ""random_strength"": 2.0, ""learning_rate"": 0.06, ""l2_leaf_reg"": 6.0, ""iterations"": 750, ""depth"": 8}","USD 23,903","USD 32,980","USD 28,441","USD 37,651","USD 53,592",0.596,0.649,"USD 22,763",14.5%,0.75%,7e8cf9d2352e4b7c8c0ea3b6593018bb
5,random_forest,random_forest__03,"{""n_estimators"": 300, ""min_samples_leaf"": 4, ""max_samples"": 0.75, ""max_features"": 0.6, ""max_depth"": 9}","USD 24,586","USD 34,034","USD 29,310","USD 38,259","USD 53,307",0.583,0.653,"USD 22,618",13.3%,1.08%,43d953d1049445b2aa88164b08121ca3
6,ridge,ridge__01,"{""alpha"": 0.1}","USD 26,758","USD 37,963","USD 32,360","USD 40,303","USD 58,566",0.537,0.581,"USD 22,546",10.8%,0.08%,ff5ce7ea96b34d108861a12bd10a2b33
7,extra_trees,extra_trees__01,"{""bootstrap"": true, ""n_estimators"": 300, ""min_samples_leaf"": 4, ""max_samples"": 0.9, ""max_features"": 0.7, ""max_depth"": 9}","USD 26,991","USD 38,152","USD 32,571","USD 40,720","USD 58,906",0.527,0.576,"USD 24,433",13.8%,0.07%,778a686db28348fdae85cf755b5f16b8
8,baseline_mediana,baseline_mediana__01,"{""strategy"": ""median""}","USD 44,702","USD 67,155","USD 55,929","USD 59,620","USD 90,517",-0.013,-0.000,"USD 37,440",11.1%,0.00%,7847df40e3c5400cbf585cf0be257cae


,modelo,configuracion,mae_promedio,mae_max,diferencia_relativa_mae
0,lightgbm,lightgbm__03,"USD 27,556","USD 32,009",0.00%
1,xgboost,xgboost__05,"USD 27,699","USD 32,179",0.52%


Modelo seleccionado para el notebook 06: lightgbm (lightgbm__03).
Criterio aplicado: equivalencia práctica del 1%, menor MAE del límite superior y compatibilidad con el pipeline productivo.
MAE promedio de validación: USD 27,556; MAE del límite superior: USD 32,009; R² mínimo/máximo: 0.609/0.669.


## 15. Conclusiones y paso al modelo definitivo

- La comparación conserva todos los resultados obtenidos y utiliza las mismas filas, variables, transformación del objetivo y partición temporal. No se eliminan modelos por su desempeño.
- El MAE promedio define el grupo de alternativas equivalentes dentro de una tolerancia del 1%. Dentro de ese grupo se prioriza el menor MAE del límite superior y, cuando corresponde, la compatibilidad con el pipeline. La aplicación de este criterio determina el modelo almacenado en `winner`.
- El notebook 06 recibe únicamente la mejor configuración seleccionada en esta etapa; no repite la búsqueda de hiperparámetros ni vuelve a comparar familias con la prueba reservada.
- Cada configuración queda registrada en el servidor definido localmente mediante `MLFLOW_TRACKING_URI`, junto con sus hiperparámetros, métricas y predicciones de validación. La dirección no se almacena en este notebook.

Después de completar y validar el notebook 06, desde la carpeta `ml/` se ejecuta:

```bash
# La configuración se lee del archivo local ml/.env
python -m ml_pipeline track
python -m ml_pipeline register-candidate
```

La versión creada se consulta desde la raíz del repositorio y solo se promueve después de revisar que quedó marcada como elegible:

```bash
set -a; source ml/.env; set +a
python model_provider/scripts/model_info.py --model salary-predictor
python model_provider/scripts/promote_model.py --model salary-predictor --version <VERSION> --alias champion --tracking-uri "$MLFLOW_TRACKING_URI"
```


In [15]:
selection = {'version':'comparacion-v4', 'modelo_seleccionado':winner,
          'configuracion_seleccionada':winner_row.configuracion,
          'criterio':'equivalencia_1pct_menor_mae_max_y_compatibilidad_pipeline',
          'razon_seleccion':selection_reason,
          'tolerancia_practica':PRACTICAL_TOLERANCE,
          'mlflow':{'tracking_uri_source':'MLFLOW_TRACKING_URI', 'experiment_name':EXPERIMENT_NAME,
                    'winner_run_id':winner_row.run_id,
                    'registered_model_name':REGISTERED_MODEL_NAME,
                    'production_alias':PRODUCTION_ALIAS},
          'mejores_por_modelo':best_by_model.to_dict(orient='records'),
          'modelos_equivalentes':eligible.to_dict(orient='records'),
          'todos_los_resultados':all_results.to_dict(orient='records'),
          'features':list(X.columns),
          'preprocesamiento':MODEL_SPECS[winner]['preprocessing'],
          'feature_contract':feature_contract.to_dict(orient='records'),
          'parametros':winner_params, 'semilla':SEED}
catalog_path = EXPERIMENT_DIR/'catalogo_hiperparametros.csv'
planned_path = EXPERIMENT_DIR/'configuraciones_planeadas.csv'
results_path = EXPERIMENT_DIR/'resultados_todas_configuraciones.csv'
best_path = EXPERIMENT_DIR/'mejor_configuracion_por_modelo.csv'
summary_path = EXPERIMENT_DIR/'resumen_experimentos.json'
selection_path = EXPERIMENT_DIR/'seleccion_modelo.json'
hyperparameter_catalog.to_csv(catalog_path,index=False)
planned_configurations.to_csv(planned_path,index=False)
all_results.to_csv(results_path,index=False)
best_by_model.to_csv(best_path,index=False)
with open(summary_path, 'w') as file:
    json.dump(selection, file, indent=2, default=str)
with open(selection_path,'w') as file:
    json.dump({'modelo_seleccionado':winner,'familia':MODEL_SPECS[winner]['family'],
               'configuracion_seleccionada':winner_row.configuracion,
               'criterio_seleccion':selection_reason,
               'tolerancia_practica':PRACTICAL_TOLERANCE,
               'parametros':winner_params,'feature_columns':list(X.columns),
               'categorical_columns':cat_cols,'numeric_columns':num_cols,
               'preprocesamiento':MODEL_SPECS[winner]['preprocessing'],
               'metricas_validacion':{key:winner_row[key] for key in ['mae_min','mae_max','mae_promedio',
                   'rmse_min','rmse_max','r2_min','r2_max','mae_amplitud','cobertura_intervalo']},
               'espacio_busqueda':MODEL_SPECS[winner]['search'],
               'mlflow_tracking_uri_source':'MLFLOW_TRACKING_URI',
               'mlflow_experiment_name':EXPERIMENT_NAME,
               'mlflow_winner_run_id':winner_row.run_id,
               'registered_model_name':REGISTERED_MODEL_NAME,
               'production_alias':PRODUCTION_ALIAS},file,indent=2,default=str)
winner_run_id = str(winner_row.run_id)
mlflow_client.set_tag(winner_run_id, 'estado_seleccion', 'ganador_validacion')
mlflow_client.set_tag(winner_run_id, 'pasa_a_notebook_06', 'true')
mlflow_client.set_tag(winner_run_id, 'criterio_seleccion', 'equivalencia_1pct_menor_mae_max')
for artifact_path in [catalog_path, planned_path, results_path, best_path, summary_path, selection_path]:
    mlflow_client.log_artifact(winner_run_id, str(artifact_path), artifact_path='seleccion_notebook_05')
print('Artefactos guardados en:', EXPERIMENT_DIR)
print('Corrida ganadora publicada en MLflow:', winner_run_id)
print('Modelo objetivo del Registry:', REGISTERED_MODEL_NAME)
print('Siguiente flujo: notebook 06 -> track -> register-candidate -> promoción aprobada a', PRODUCTION_ALIAS)


Artefactos guardados en: /Users/franciscolozano/Downloads/PROYECTOS_MAIA/PROYECTO_DE_GRADO/MICROPROYECTO_GITHUB/ml/artifacts/comparacion_modelos
Corrida ganadora publicada en MLflow: 4018203a81b7420da58caacea28c2d9b
Modelo objetivo del Registry: salary-predictor
Siguiente flujo: notebook 06 -> track -> register-candidate -> promoción aprobada a champion
